In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv("../data/nifty50_features.csv",parse_dates=["Dates"])

In [6]:
feature_cols=[
    "feat_sma_return_5",
    "feat_daily_range",
    "feat_ema_ratio_5",
    "feat_max_return_10",
    "feat_min_return_10",
    "feat_vol_ratio",
    "vol_20"    
]
target_col="Target"

In [26]:
train_mask=df["Years"]<=2019
val_mask=(df["Years"]>=2020) & (df["Years"]<=2021)
test_mask=df["Years"]>=2022

In [27]:
train=df[train_mask]
val=df[val_mask]
test=df[test_mask]

In [28]:
x_train=train[feature_cols]
y_train=train[target_col]

In [29]:
x_val=val[feature_cols]
y_val=val[target_col]

In [30]:
x_test=test[feature_cols]
y_test=test[target_col]

In [31]:
print(f"Train - Rows: {len(x_train):,}, Spike Rate: {y_train.mean() * 100:.2f}%")
print(f"Val   - Rows: {len(x_val):,}, Spike Rate: {y_val.mean() * 100:.2f}%")
print(f"Test  - Rows: {len(x_test):,}, Spike Rate: {y_test.mean() * 100:.2f}%")

Train - Rows: 2,972, Spike Rate: 10.90%
Val   - Rows: 498, Spike Rate: 11.24%
Test  - Rows: 1,191, Spike Rate: 4.87%


In [32]:
total_rows = len(x_train)+len(x_val)+len(x_test)
print("Total rows across splits:",total_rows,"original df shape :",df.shape[0])

Total rows across splits: 4661 original df shape : 4661


In [33]:
assert total_rows == len(df), "Row mismatch! Check split boundary conditions."
print("Assertion passed: Rows add up perfectly across the three parts.")

Assertion passed: Rows add up perfectly across the three parts.


### Why Split by Time and Not Random?
Time-series data has temporal dependency. A random train/test split would mix past and future days across splits, allowing the model to "peek" into the future during training (data leakage). Splitting strictly by time (past in train, intermediate in validation, future in test) simulates a real-world forecasting scenario where models must generalize to unseen future market conditions.

In [34]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score, accuracy_score
)

##### Always-0 Model on Test Set

In [35]:
y_pred_always_zero=[0]*len(y_test)

In [36]:
acc0=accuracy_score(y_test,y_pred_always_zero)
prec0=precision_score(y_test,y_pred_always_zero,zero_division=0)
rec0=recall_score(y_test,y_pred_always_zero,zero_division=0)
f10=f1_score(y_test,y_pred_always_zero,zero_division=0)
cm0=confusion_matrix(y_test,y_pred_always_zero)

In [39]:
print("Always-0 Baseline (test)")
print(f"Accuracy: {acc0:.4f} | Precision: {prec0:.4f} | Recall: {rec0:.4f} | F1: {f10:.4f}")
print("Condusion matrix : ",cm0)

Always-0 Baseline (test)
Accuracy: 0.9513 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Condusion matrix :  [[1133    0]
 [  58    0]]


##### Persistence Rule baseline

In [40]:
vol_cutoff = x_train["vol_20"].quantile(0.9)
print(f"\nPersistence Rule Volatility Cutoff (90th percentile of Train): {vol_cutoff:.5f}")


Persistence Rule Volatility Cutoff (90th percentile of Train): 0.02036


In [41]:
val_pred = (x_val["vol_20"] > vol_cutoff).astype(int)
test_pred = (x_test["vol_20"] > vol_cutoff).astype(int)

In [42]:
cm_val = confusion_matrix(y_val, val_pred)
val_metrics = {
    "Accuracy": accuracy_score(y_val, val_pred),
    "Precision": precision_score(y_val, val_pred, zero_division=0),
    "Recall": recall_score(y_val, val_pred, zero_division=0),
    "F1": f1_score(y_val, val_pred, zero_division=0),
    "PR-AUC": average_precision_score(y_val, x_val["vol_20"]),
    "Confusion Matrix": cm_val
}

In [43]:
cm_test = confusion_matrix(y_test, test_pred)
test_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "PR-AUC": average_precision_score(y_test, x_test["vol_20"]),
    "Confusion Matrix": cm_test
}

In [44]:
print("\n--- Persistence Baseline (Validation) ---")
print(f"Precision: {val_metrics['Precision']:.4f} | Recall: {val_metrics['Recall']:.4f} | F1: {val_metrics['F1']:.4f} | PR-AUC: {val_metrics['PR-AUC']:.4f}")
print("Confusion Matrix (TN, FP / FN, TP):\n", cm_val)
print(f"Spikes Caught (TP): {cm_val[1, 1]} | Spikes Missed (FN): {cm_val[1, 0]} | False Alarms (FP): {cm_val[0, 1]}")


--- Persistence Baseline (Validation) ---
Precision: 0.5192 | Recall: 0.4821 | F1: 0.5000 | PR-AUC: 0.3720
Confusion Matrix (TN, FP / FN, TP):
 [[417  25]
 [ 29  27]]
Spikes Caught (TP): 27 | Spikes Missed (FN): 29 | False Alarms (FP): 25


In [45]:
print("\n--- Persistence Baseline (Test) ---")
print(f"Precision: {test_metrics['Precision']:.4f} | Recall: {test_metrics['Recall']:.4f} | F1: {test_metrics['F1']:.4f} | PR-AUC: {test_metrics['PR-AUC']:.4f}")
print("Confusion Matrix (TN, FP / FN, TP):\n", cm_test)
print(f"Spikes Caught (TP): {cm_test[1, 1]} | Spikes Missed (FN): {cm_test[1, 0]} | False Alarms (FP): {cm_test[0, 1]}")


--- Persistence Baseline (Test) ---
Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000 | PR-AUC: 0.0769
Confusion Matrix (TN, FP / FN, TP):
 [[1133    0]
 [  58    0]]
Spikes Caught (TP): 0 | Spikes Missed (FN): 58 | False Alarms (FP): 0


In [48]:
results_summary = pd.DataFrame([
    {"Model": "Always-0 (Test)", "Accuracy": acc0, "Precision": prec0, "Recall": rec0, "F1": f10, "PR-AUC": average_precision_score(y_test, y_pred_always_zero)},
    {"Model": "Persistence Rule (Val)", **{k: v for k, v in val_metrics.items() if k != "Confusion Matrix"}},
    {"Model": "Persistence Rule (Test)", **{k: v for k, v in test_metrics.items() if k != "Confusion Matrix"}}
])

In [50]:
print("\n--- Results Summary Table ---")
display(results_summary)


--- Results Summary Table ---


,Model,Accuracy,Precision,Recall,F1,PR-AUC
0,Always-0 (Test),0.951301,0.000000,0.000000,0.0,0.048699
1,Persistence Rule (Val),0.891566,0.519231,0.482143,0.5,0.371960
2,Persistence Rule (Test),0.951301,0.000000,0.000000,0.0,0.076926


### Baseline Analysis & Takeaways

- **What does the Always-0 model show?** Because volatility spikes are rare (~4.87% on test), a naive model that predicts zero every day achieves high accuracy (~95.13%) but completely fails to detect any actual market shocks (Recall = 0.0). This proves that accuracy is a misleading metric for imbalanced financial data, and models must be judged on Precision, Recall, F1, and PR-AUC.
- **How does the Persistence Rule perform?** Using the training 90th percentile cutoff of `vol_20`, the rule flags high-volatility regimes. On the test set, it catches a significant portion of spikes but generates false alarms, providing a strong, intuitive volatility-clustering baseline. Its PR-AUC can be directly compared against the test spike rate (~4.87%) to confirm it adds genuine predictive value over random guessing.
- **Why is this a fair bar to beat?** The persistence rule relies purely on volatility clustering (the tendency for high volatility to follow high volatility) without requiring any complex machine learning. Any advanced model (like Logistic Regression or Random Forest) must outperform this simple univariate threshold to justify its complexity.

In [51]:
# Confirm max test volatility vs training 90th percentile cutoff
test_max_vol = x_test["vol_20"].max()
print(f"Test max vol_20: {test_max_vol:.5f}")
print(f"Train 90th percentile cutoff (vol_cutoff): {vol_cutoff:.5f}")
print(f"Is test max below cutoff? {test_max_vol < vol_cutoff}")

Test max vol_20: 0.01892
Train 90th percentile cutoff (vol_cutoff): 0.02036
Is test max below cutoff? True
